In [11]:
import torch
from torch import nn
import torch.optim as optim
import numpy as np
import random

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)

Using device: cuda


читаем текст и создаем словарь

In [12]:
def read_text(path):
    with open(path, "r", encoding="utf-8") as f:
        return f.read() #текст превращаем в одну длинную строку

text = read_text("tinyshakespeare.txt")

chars = sorted(list(set(text))) #только уникальные символы превращаем в список и сортируем
vocab_size = len(chars) #размер словаря

char2idx = {ch: i for i, ch in enumerate(chars)} #из символа в число
idx2char = {i: ch for i, ch in enumerate(chars)} #из числа в символ

print("Vocab size:", vocab_size)

Vocab size: 65


текст -> числа

In [13]:
encoded = np.array([char2idx[ch] for ch in text]) #каждый символ текста заменяем на число

подготовим данные

In [14]:
SEQ_LEN = 40 #будем предсказывать символы на сонове 40 предыдущих

def create_batches(data, seq_len):
    X, y = [], []
    for i in range(len(data) - seq_len):
        X.append(data[i:i+seq_len])
        y.append(data[i+1:i+seq_len+1]) #то что в Х но сдвинуто на 1
    return np.array(X), np.array(y)

X, y = create_batches(encoded, SEQ_LEN)

split = int(0.9 * len(X))
X_train, X_test = X[:split], X[split:]
y_train, y_test = y[:split], y[split:]

RNN модель

In [15]:
class CharRNN(nn.Module):
    def __init__(self, vocab_size, embedding_size, hidden_size):
        super().__init__()

        self.embedding = nn.Embedding(vocab_size, embedding_size) #номер символа в вектор
        self.rnn = nn.RNN(embedding_size, hidden_size, batch_first=True)
        self.fc = nn.Linear(hidden_size, vocab_size) #скрытое состояние в символ

    def forward(self, x, h=None):
        x = self.embedding(x)
        out, h = self.rnn(x, h)
        out = self.fc(out)
        return out, h

обучаем

In [16]:
#размеры слоев
EMBEDDING_SIZE = 32
HIDDEN_SIZE = 128

model = CharRNN(vocab_size, EMBEDDING_SIZE, HIDDEN_SIZE).to(device)

loss_fn = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=0.001)

BATCH_SIZE = 64
EPOCHS = 30

for epoch in range(EPOCHS):
    model.train()
    total_loss = 0

    for i in range(0, len(X_train), BATCH_SIZE):
        xb = torch.tensor(X_train[i:i+BATCH_SIZE]).to(device)
        yb = torch.tensor(y_train[i:i+BATCH_SIZE]).to(device)

        optimizer.zero_grad()

        outputs, _ = model(xb)

        loss = loss_fn(
            outputs.reshape(-1, vocab_size),
            yb.reshape(-1)
        )

        loss.backward()
        optimizer.step()

        total_loss += loss.item()

    print(f"Epoch {epoch+1}/{EPOCHS}, Loss: {total_loss:.4f}")

Epoch 1/30, Loss: 29014.6923
Epoch 2/30, Loss: 26190.1836
Epoch 3/30, Loss: 25514.3289
Epoch 4/30, Loss: 25203.6267
Epoch 5/30, Loss: 25019.7794
Epoch 6/30, Loss: 24886.3813
Epoch 7/30, Loss: 24787.7466
Epoch 8/30, Loss: 24722.0978
Epoch 9/30, Loss: 24667.5524
Epoch 10/30, Loss: 24623.8974
Epoch 11/30, Loss: 24592.7971
Epoch 12/30, Loss: 24571.6745
Epoch 13/30, Loss: 24539.9088
Epoch 14/30, Loss: 24528.7743
Epoch 15/30, Loss: 24517.1843
Epoch 16/30, Loss: 24507.0816
Epoch 17/30, Loss: 24494.4140
Epoch 18/30, Loss: 24484.5051
Epoch 19/30, Loss: 24479.7325
Epoch 20/30, Loss: 24461.7425
Epoch 21/30, Loss: 24457.2436
Epoch 22/30, Loss: 24448.2455
Epoch 23/30, Loss: 24460.1275
Epoch 24/30, Loss: 24453.4355
Epoch 25/30, Loss: 24460.9447
Epoch 26/30, Loss: 24449.9965
Epoch 27/30, Loss: 24440.1891
Epoch 28/30, Loss: 24449.8734
Epoch 29/30, Loss: 24444.3692
Epoch 30/30, Loss: 24444.2450


генерация текста

In [18]:
def generate_text(model, start_text, length=1000):
    model.eval()

    input_seq = torch.tensor(
        [[char2idx[ch] for ch in start_text]],
        dtype=torch.long
    ).to(device)

    h = None
    generated = start_text

    for _ in range(length):
        outputs, h = model(input_seq, h)
        last_logits = outputs[0, -1]

        probs = torch.softmax(last_logits, dim=0)
        idx = torch.multinomial(probs, 1).item()

        generated += idx2char[idx]
        input_seq = torch.tensor([[idx]]).to(device)

    return generated

seed = "First Citizen:\n"
print(generate_text(model, seed))

First Citizen:
You be more; thether wrong the humoortu' I come ano'd made, been and my denest well gentlemate,
Provost.

KING:
Why be worst a constan, to live fair'd me wrongs upon thee!

HORTENSIO:
If you moar
That she deptrele, I tif, sealoes, when you have Hengend whe most dark the servant this tresship of such canst down?

HORTENSIO:
'Tis not be is't:
If I so prison?
On sting you not die
And my great me and be abeections to the sirdisal but this sits? Well. Belows! my long
Upon me denied he's eldish'll.

KLO:
Be cave to here my well.

POMPEY:
What, as Kind and love to gration of ever yourst where noble be mistred to'tr King Bittleds you be valousle of the porsand do you and that I be
Forthe heary.

KING OVEn you did, Banking, fhook he then,
That jeasue of once, go me.
Ald, be paptenne't death, stabe:
nee, sweet not death.
You him, I'll thou wans
Pauthion when hast the gandss is
step, that,
Knorishew me would enouge gassenges wood this and you had will, her instructer.

KATHe mine t